# Week 1 Summary - BSM Foundations + First Real SPX Smile (Broken, need fix)

## What was built

Three independent European option pricers, all agreeing on the canonical BSM ATM 1Y σ=20% r=5% call value (10.4506) within their respective error bars:

1. **Closed-form BSM** (`models/bsm.py`) - vectorised, broadcasts over 2D grids of strikes × maturities. Includes analytic Greeks (Δ, Γ, 𝒱, Θ, ρ), verified against finite-difference Greeks to <1e-6 and against the BSM PDE identity to machine epsilon (2e-16).

2. **CRR binomial tree** (`models/binomial.py`) - O(N) memory, O(N²) time. Empirically observed O(1/N) convergence to BSM. Strike-discretisation oscillation visible at OTM strikes. Richardson extrapolation gives O(1/N²) on smooth (ATM) cases, ~1000× error reduction at N=500.

3. **Monte Carlo on GBM** (`models/montecarlo.py`) - direct sampling from closed-form GBM (no time-stepping bias). Vectorised, reproducible via seed. Returns (price, standard_error). O(1/√N) convergence verified over 4 orders of magnitude. Antithetic variates achieve theoretical 2× variance reduction (SE ratio 1/√2 ≈ 0.707) on monotone vanilla payoffs.

Plus two infrastructure modules:

4. **Implied vol inversion** (`models/implied_vol.py`) - Brent's method round-tripping to ~1e-13 on synthetic prices. Returns `np.nan` for prices violating no-arbitrage bounds.

5. **Option chain data hygiene** (`models/option_chain.py`) - yfinance and CBOE fetchers with file-based caching. CBOE preferred (real OI data, fresher quotes, full chain depth). Filters bid > 0, spread, volume, OI, staleness. Decodes OCC contract symbols, distinguishes SPXW (PM-settled weekly) from SPX (AM-settled monthly) - necessary because both series trade simultaneously at most expiries.

## Empirical findings

**The SPX volatility smile is real and the equity skew is steep.**

30 DTE smile (expiry 2026-07-17, spot = 7511):
- ATM IV ≈ 14.7%
- Smile range: 11.3% (OTM) to 17.9% (ITM) → 6.6 vol-point range
- Minimum at log-moneyness ≈ +0.05 (5% OTM)
- Left wing (ITM calls / equiv OTM puts) steeper than right wing — equity skew confirmed

75 DTE smile (expiry 2026-08-31):
- ATM IV ≈ 14.7% - flat ATM term structure (calm regime)
- Smile range: 12.2% to 15.9% → 3.7 vol-point range
- Notably **flatter than the 30 DTE smile** at every comparable strike

**Term-structure flattening empirically confirmed**: the per-unit-strike skew at 30 DTE is roughly 70% steeper than at 75 DTE. This is precisely the empirical fact that Markovian stochastic vol models (Heston) struggle to reproduce, and that rough volatility models were largely motivated to explain.

## Pedagogical bugs caught (for the record)

- **CRR strike discretisation**: errors flip sign with parity of N at OTM strikes. Tree-node grid wobbles relative to fixed K as N changes.
- **MC antithetic variates, attempt 1**: concatenation order paired non-antithetic samples → 0× reduction.
- **MC antithetic variates, attempt 2**: wrong reshape axis → spurious 590,000× "reduction" (caught via z-score: estimate was 273 SEs from truth → impossible).
- **yfinance openInterest column**: always 0 for SPX index options. Discovered when OI filter killed every row. Default OI filter set to 0 for yfinance source; CBOE provides real OI.
- **CBOE duplicate strikes**: SPX and SPXW are separate contracts. Parser updated to extract `root`; downstream filter to SPXW only.

## What this sets up

The shape of these smiles is the empirical motivation for Phase 1 weeks 3-9. BSM cannot generate this smile (σ is constant). Local volatility (Week 3) can fit a snapshot but not predict the next one. Heston (Week 4) generates a smile through stochastic σ² but flattens too quickly with maturity. Jumps (Week 7) add the fat tails. Rough Bergomi (Week 9) reproduces the steep short-dated ATM skew that Markovian models cannot. Each model is a more refined attempt to explain the shape and dynamics you see above.

## Open issues / Week 2 starting points

- **Risk-free rate**: hardcoded at 4.5%. Should be pulled from FRED (3-month T-bill or SOFR curve) per maturity for accuracy.
- **Dividend yield**: SPX has ~1.3% dividend yield, not currently modelled. Affects forward, which affects IV inversion.
- **Smile smoothing**: noisy wing IVs propagate from bid-ask spread. SVI parameterisation in Week 3.
- **Greeks: pathwise & likelihood-ratio**: Week 2 main topic.
- **GARCH(1,1)**: realised vs implied vol comparison. Week 2 secondary topic.
- **Data hygiene**: extract OI filter + spread filter defaults into a per-source config rather than function defaults.

In [2]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
from models.bsm import bsm_price
from models.option_chain import (
    fetch_chain_cboe,
    filter_by_expiry,
    clean_chain,
    find_closest_expiry,
    implied_forward_from_parity,
    
)
from models.implied_vol import (
    bsm_implied_vol,
    compute_smile,
)

In [3]:
# Generate a known price
sigma_true = 0.25
C = bsm_price(100, 100, 1.0, 0.05, sigma_true, 'call')

# Invert
sigma_implied = bsm_implied_vol(C, 100, 100, 1.0, 0.05, 'call')

print(f"True sigma:     {sigma_true}")
print(f"Implied sigma:  {sigma_implied}")
print(f"Difference:     {abs(sigma_true - sigma_implied):.2e}")

True sigma:     0.25
Implied sigma:  0.24999999999982547
Difference:     1.75e-13


In [4]:
# OTM call
sigma_true = 0.25
C = bsm_price(100, 110, 1.0, 0.05, sigma_true, 'call')
sigma_imp = bsm_implied_vol(C, 100, 110, 1.0, 0.05, 'call')
print(f"OTM call: true={sigma_true}, implied={sigma_imp}, diff={abs(sigma_true - sigma_imp):.2e}")

# OTM put
P = bsm_price(100, 90, 1.0, 0.05, sigma_true, 'put')
sigma_imp = bsm_implied_vol(P, 100, 90, 1.0, 0.05, 'put')
print(f"OTM put:  true={sigma_true}, implied={sigma_imp}, diff={abs(sigma_true - sigma_imp):.2e}")

OTM call: true=0.25, implied=0.24999999999998523, diff=1.48e-14
OTM put:  true=0.25, implied=0.24999999870277942, diff=1.30e-09


In [5]:
# A clearly invalid call price (below intrinsic)
result = bsm_implied_vol(0.01, 100, 100, 1.0, 0.05, 'call')
print(f"Below intrinsic: {result}")  # Should print 'nan'

# A clearly invalid call price (above S)
result = bsm_implied_vol(150, 100, 100, 1.0, 0.05, 'call')
print(f"Above upper bound: {result}")  # Should print 'nan'

Below intrinsic: nan
Above upper bound: nan


In [7]:
df, spot = fetch_chain_cboe('^SPX')
# print(f"Calls: {df.shape}, Puts: {df.shape}")
# print(f"\nColumns: {calls.columns.tolist()}")
# print(f"\nFirst 5 calls:")
# print(calls.head())
# print(f"\nLast 5 calls:")
# print(calls.tail())

In [8]:
print(df)
print(spot)

                    option     bid  bid_size      ask  ask_size      iv  \
0       SPX260717C00200000  7235.7       1.0  7243.50       1.0  0.0000   
1       SPX260717P00200000     0.0       0.0     0.05     121.0  4.5224   
2       SPX260717C00400000  7036.2       1.0  7044.00       1.0  0.0000   
3       SPX260717P00400000     0.0       0.0     0.05     117.0  3.6192   
4       SPX260717C00600000  6836.8       1.0  6844.30       1.0  0.0000   
...                    ...     ...       ...      ...       ...     ...   
30463  SPXW270630P09300000  1531.0       1.0  1571.00       1.0  0.1349   
30464  SPXW270630C09400000    36.2      75.0    37.30      75.0  0.1363   
30465  SPXW270630P09400000  1619.1       1.0  1659.10       1.0  0.1341   
30466  SPXW270630C09600000    25.1      82.0    26.10      82.0  0.1359   
30467  SPXW270630P09600000  1798.9       1.0  1838.80       1.0  0.1328   

       openInterest  volume   delta   gamma  ...      low       tick  \
0           14745.0     1.0

In [ ]:
print("Volume statistics for calls:")
print(calls['volume'].describe())
print(f"\nNumber of calls with volume = 0 or NaN: {(calls['volume'].fillna(0) == 0).sum()} / {len(calls)}")
print(f"Number of calls with volume >= 10: {(calls['volume'] >= 10).sum()} / {len(calls)}")
print(f"\nBid/Ask spread analysis (calls):")
spread_pct = (calls['ask'] - calls['bid']) / ((calls['ask'] + calls['bid']) / 2)
print(spread_pct.describe())

In [ ]:
print(f"\nStrike range: {calls['strike'].min()} to {calls['strike'].max()}")
print(f"Spot: 7554.29")
print(f"\nNumber of strikes: {len(calls)}")

In [ ]:
calls_clean = clean_chain(
    calls,
    min_volume=5,
    min_open_interest=0,
    max_relative_spread=0.5,
    max_staleness_days=1,
    verbose=True,
)
print(f"\nFinal: {len(calls_clean)} rows")
print(calls_clean[['strike', 'bid', 'ask', 'mid', 'volume', 'lastTradeDate']].sort_values('strike'))

In [ ]:
# Run it
smile = compute_smile(
    calls_clean,
    spot=7554.29,
    T=30/365,
    r=0.045,
    option_type='call',
)
print(smile[['strike', 'mid', 'iv']].to_string())

In [ ]:
# Compute log-moneyness for the x-axis: log(K/F) where F = S * exp(rT)
spot = 7554.29
r = 0.045
T = 30/365
F = spot * np.exp(r * T)

smile['log_moneyness'] = np.log(smile['strike'] / F)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: IV vs strike
axes[0].plot(smile['strike'], smile['iv'] * 100, 'o-', color='steelblue')
axes[0].axvline(spot, color='red', linestyle='--', alpha=0.5, label=f'Spot = {spot}')
axes[0].set_xlabel('Strike')
axes[0].set_ylabel('Implied Volatility (%)')
axes[0].set_title('SPX call implied vol vs strike (30 DTE, expiry 2026-07-16)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: IV vs log-moneyness - the conventional academic plot
axes[1].plot(smile['log_moneyness'], smile['iv'] * 100, 'o-', color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5, label='ATM forward')
axes[1].set_xlabel('Log-moneyness  log(K/F)')
axes[1].set_ylabel('Implied Volatility (%)')
axes[1].set_title('Same smile, log-moneyness axis')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Switching to CBOE as Yfinance is not returning anything!
all_options, spot = fetch_chain_cboe('^SPX')
print(f"Total contracts: {len(all_options)}")
print(f"Unique expiries: {all_options['expiry'].nunique()}")
print(f"First 10 expiries: {sorted(all_options['expiry'].unique())[:10]}")
print(all_options['root'].value_counts())   # see what's there
spxw = all_options[all_options['root'] == 'SPXW']
print(f"After filtering to SPXW: {len(spxw)} contracts")

today = pd.Timestamp.now(tz='UTC').normalize()
expiries = sorted(spxw['expiry'].unique())

expiry_30, days_30 = find_closest_expiry(expiries, target_days=30)
expiry_75, days_75 = find_closest_expiry(expiries, target_days=75)

T_30 = days_30 / 365
T_75 = days_75 / 365

print(f"30 DTE target → {expiry_30} ({days_30} days out)")
print(f"60 DTE target → {expiry_75} ({days_75} days out)")
print(f"Current Spot: {spot}")

today = today.tz_localize(None)

print(today)
print(today.tzinfo)

spxw['expiry'] = pd.to_datetime(spxw['expiry']).dt.tz_localize(None)

candidates = [
    e for e in sorted(spxw['expiry'].unique())
    if e > today + pd.Timedelta(days=50)
    and e < today + pd.Timedelta(days=80)
]

for e in candidates:
    n = (spxw['expiry'] == e).sum()
    days = (e - today).days
    print(f"{e.date()} ({days} DTE): {n} contracts listed")

The code is picking the closest available to 30 and 75 DTE.

In [ ]:
r = 0.045

# 30 DTE
calls_30, _ = filter_by_expiry(spxw, expiry_30)
calls_30_clean = clean_chain(
    calls_30,
    min_volume=5,
    min_open_interest=10,
    max_relative_spread=0.30,
    max_staleness_days=2,
)
smile_30 = compute_smile(calls_30_clean, spot=spot, T=T_30, r=r, option_type='call')

# 75 DTE  
calls_75, _ = filter_by_expiry(all_options, expiry_75)
calls_75_clean = clean_chain(
    calls_75,
    min_volume=5,
    min_open_interest=10,
    max_relative_spread=0.30,
    max_staleness_days=2,
)
smile_75 = compute_smile(calls_75_clean, spot=spot, T=T_75, r=r, option_type='call')

print(f"30 DTE: {len(smile_30)} usable strikes")
print(f"75 DTE: {len(smile_75)} usable strikes")
print()
print("=== 30 DTE smile ===")
print(smile_30[['strike', 'mid', 'iv']].to_string())
print()
print("=== 75 DTE smile ===")
print(smile_75[['strike', 'mid', 'iv']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Forward prices for each expiry
F_30 = spot * np.exp(r * T_30)
F_75 = spot * np.exp(r * T_75)

smile_30['log_moneyness'] = np.log(smile_30['strike'] / F_30)
smile_75['log_moneyness'] = np.log(smile_75['strike'] / F_75)

# Left plot: IV vs strike
axes[0].plot(smile_30['strike'], smile_30['iv'] * 100, 'o-', color='steelblue', label=f'30 DTE')
axes[0].plot(smile_75['strike'], smile_75['iv'] * 100, 's-', color='darkorange', label=f'75 DTE')
axes[0].axvline(spot, color='red', linestyle='--', alpha=0.4, label=f'Spot = {spot:.0f}')
axes[0].set_xlabel('Strike')
axes[0].set_ylabel('Implied Volatility (%)')
axes[0].set_title('SPX call smiles: 30 DTE vs 75 DTE')
axes[0].set_xlim(7000, 8500)  # zoom in; 6000 outlier distorts the plot
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right plot: IV vs log-moneyness
axes[1].plot(smile_30['log_moneyness'], smile_30['iv'] * 100, 'o-', color='steelblue', label='30 DTE')
axes[1].plot(smile_75['log_moneyness'], smile_75['iv'] * 100, 's-', color='darkorange', label='75 DTE')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.4, label='ATM forward')
axes[1].set_xlabel('Log-moneyness  log(K/F)')
axes[1].set_ylabel('Implied Volatility (%)')
axes[1].set_title('Same smiles, log-moneyness axis')
axes[1].set_xlim(-0.10, 0.15)  # zoom; exclude outliers
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

Term-structure flattening is unmistakable, the 75 DTE curve sits noticeably above 30 DTE on the right wing (less depressed) and the left wing of 30 DTE shoots up sharply while 75 DTE rises gently. Textbook result shown with real data.

In [ ]:
# 1. you need BOTH calls and puts for the same expiry (parity needs the pair)
calls_30, puts_30 = filter_by_expiry(spxw, expiry_30)
calls_30_clean = clean_chain(calls_30, source="cboe")
puts_30_clean  = clean_chain(puts_30,  source="cboe")

# 2. extract the parity forward
F, parity_table = implied_forward_from_parity(calls_30_clean, puts_30_clean, r=0.045, T=T_30)

# 3. convert to carry, report the implied dividend yield as a sanity check
b = np.log(F / spot) / T_30
q_implied = 0.045 - b
print(f"spot {spot:.2f}, parity forward {F:.2f}, implied q = {q_implied:.2%}")

# 4. re-extract the smile, now dividend-correct
smile_30 = compute_smile(calls_30_clean, spot=spot, T=T_30, r=0.045, b=b, option_type="call")

In [ ]:
print(parity_table[['strike', 'call_mid', 'put_mid', 'F_est']].to_string())

In [ ]:
print(f"CBOE current_price: {spot}")
# what does the chain itself think? the ATM strike (where |C-P| smallest) ~ forward
parity_table['CP_gap'] = (parity_table['call_mid'] - parity_table['put_mid']).abs()
print(parity_table.loc[parity_table['CP_gap'].idxmin()])